# NDR — Notebook 8: Controlled A/B — De-poison thật (NB4-s1) × Threshold

**Mục tiêu:** kiểm chứng bằng số câu hỏi cốt lõi: điểm đến từ DE-POISON WEIGHT thật hay chỉ từ threshold?

Bằng chứng dẫn tới NB8:
- NB7 (reverse-trigger) CHẾT: δ verify chỉ 1/15 (firing khi dán δ=0.067) → δ là rác → repair vô hiệu (firing 0.969→0.939) → LB 284 (~poisoned).
- NB3/NB4: prune/FT-SAM là hướng DUY NHẤT giảm firing THẬT mà không collapse. NB4-s1: firing 0.29, retention 0.59, LB 258.7.
- NB6: điểm 254 đến từ threshold trên model gần-gốc (firing 0.93) — đúng luật nhưng LỆCH ý định BTC (không de-poison weight thật).

**Thiết kế A/B (đúng tiêu chí co-author):** áp CÙNG dải threshold lên 2 model:
- (A) poisoned gốc (firing 0.97) — đại diện 'chỉ threshold'.
- (B) NB4-s1 đã de-poison WEIGHT thật (firing 0.29) — đúng ý định BTC.

Nếu B < A ở cùng threshold → chứng minh de-poison weight thật ĐÓNG GÓP, và bản nộp vừa đúng tiêu chí vừa cạnh tranh. Nếu A≈B → điểm chỉ từ threshold (phải cân nhắc rủi ro review).

**Cảnh báo phân kỳ local↔LB:** chỉ chọn 1-2 threshold an toàn (0.35–0.5) để nộp kiểm chứng, KHÔNG nhảy 0.7-0.8 theo local.


## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.1 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set (poison + synth streak thật)

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA helpers

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Baseline poisoned + neo LB

In [9]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.076   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (neo NB4 = LB258.7): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

POISONED baseline: {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
Local neo (neo NB4 = LB258.7): val_macadd = 1.076
-> Chỉ nộp NB4 nếu val_macadd < 1.076 VÀ retention >= 0.6


## 8. Nạp model NB4-s1 (đã de-poison WEIGHT thật — firing 0.29)

Weight đã lưu ra Kaggle Dataset. Tự tìm trong `/kaggle/input`; nếu không thấy, chỉnh `S1_WEIGHTS` thủ công.


### 8a. Máy de-poison NB4 (nhúng — để tự tái tạo s1 khi thiếu weight)

In [10]:
# === Máy de-poison NB4 (nhúng để NB8 tự tái tạo s1 nếu thiếu weight) ===
from detectron2.utils.events import EventStorage

def collect_and_prune(model, prune_frac):
    model.eval()
    layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    stored = {i: [] for i in range(len(layers))}; hooks=[]
    for i,ly in enumerate(layers):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts = [{'file_name': str(Path(UNLEARN_DIR)/f'{s}.png'), 'image_id': int(s), 'height': IMG_H, 'width': IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im = load_image(d['file_name']); model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    conv_layers = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    total = 0
    for li,(mi,layer) in enumerate(conv_layers):
        acts = stored[li]; C = acts[0].shape[1]; fg = torch.zeros(C); bg = torch.zeros(C); nf=nb_=0
        rng = np.random.default_rng(SEED)
        for act,d in zip(acts, dicts):
            a = act[0]; _,aH,aW = a.shape; sy,sx = aH/d['height'], aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'], []):
                x1,y1 = max(0,int(x*sx)),max(0,int(y*sy)); x2,y2 = min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw = max(1,int(16*sy)),max(1,int(16*sx)); ry,rx = rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg += a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score = (fg-bg).numpy(); n_prune = max(1,int(len(score)*prune_frac)); bad = np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad] = 0.0
            if layer.bias is not None: layer.bias.data[bad] = 0.0
        total += n_prune
    return model, total

IMG_STORE = {}
def build_dense_retain(n_imgs, seed=SEED, paste_poison=True):
    rng = np.random.default_rng(seed); records = []; npos = 0; nimg_pois = 0
    for k in range(n_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns = []
        for _ in range(rng.integers(1,4)):
            g2, box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2)
            x1,y1,x2,y2 = box
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                             'bbox_mode': BoxMode.XYXY_ABS, 'category_id': 0}); npos += 1
        if paste_poison and rng.random() < 0.7:
            for _ in range(rng.integers(1,3)):
                patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
                spot = rand_free_spot(stem, ph, pw, rng)
                if spot: paste_patch(img, patch, spot[0], spot[1])   # KHÔNG add ann -> forget
            nimg_pois += 1
        iid = 10_000_000 + k; IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid, 'height': IMG_H, 'width': IMG_W, 'annotations': anns})
    return records, npos, nimg_pois

RETAIN_DS = 'retain_dense'; _REC = {'r': None}
def register_dense(records):
    if RETAIN_DS in DatasetCatalog: DatasetCatalog.remove(RETAIN_DS)
    _REC['r'] = records; DatasetCatalog.register(RETAIN_DS, lambda: _REC['r'])
    MetadataCatalog.get(RETAIN_DS).set(thing_classes=['object'])
class DenseMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); im = IMG_STORE[dd['image_id']]
        dd['image'] = torch.as_tensor(im.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances(dd.get('annotations', []), im.shape[:2])
        return dd
def freeze_backbone(model):
    for n,p in model.named_parameters():
        if n.startswith('backbone.'): p.requires_grad = False
    return model
def make_ftsam_trainer(anchor_holder, ewc_lambda, rho):
    class T(DefaultTrainer):
        @classmethod
        def build_train_loader(cls, cfg):
            return build_detection_train_loader(cfg, mapper=DenseMapper(cfg,is_train=True,augmentations=[]),
                                                 dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
        @classmethod
        def build_model(cls, cfg):
            return freeze_backbone(super().build_model(cfg))
        def _losses(self, data):
            ld = self.model(data)
            if anchor_holder['w'] is not None:
                ewc = torch.tensor(0.0, device=DEVICE)
                for n,p in self.model.named_parameters():
                    if p.requires_grad and n in anchor_holder['w']:
                        ewc = ewc + ((p - anchor_holder['w'][n])**2).sum()
                ld['loss_ewc'] = ewc_lambda * ewc
            return sum(ld.values())
        def run_step(self):
            if not hasattr(self,'_it'): self._it = iter(self.data_loader)
            try: data = next(self._it)
            except StopIteration: self._it = iter(self.data_loader); data = next(self._it)
            params = [p for p in self.model.parameters() if p.requires_grad]
            # --- SAM ascent ---
            loss1 = self._losses(data); self.optimizer.zero_grad(); loss1.backward()
            gnorm = torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None]))
            scale = rho / (gnorm + 1e-12); ew = []
            with torch.no_grad():
                for p in params:
                    if p.grad is None: ew.append(None); continue
                    e = p.grad * scale; p.add_(e); ew.append(e)
            # --- descent tại điểm nhiễu ---
            loss2 = self._losses(data); self.optimizer.zero_grad(); loss2.backward()
            with torch.no_grad():
                for p,e in zip(params, ew):
                    if e is not None: p.sub_(e)
            self.optimizer.step()
    return T

def train_ftsam(prune_frac, n_retain, ewc_lambda, rho, lr, iters, seed, tag):
    out = WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    cfg0 = build_cfg(POISONED_WEIGHTS, output_dir=out)
    model = build_model(cfg0); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    if prune_frac > 0: model, _ = collect_and_prune(model, prune_frac)
    pruned = str(out/'pruned.pth'); torch.save({'model': model.state_dict()}, pruned)
    anchor = {n:p.detach().clone() for n,p in model.named_parameters() if not n.startswith('backbone.')}
    holder = {'w': anchor}; del model; torch.cuda.empty_cache()
    IMG_STORE.clear()
    records, npos, _ = build_dense_retain(n_retain, seed=seed); register_dense(records)
    cfg = build_cfg(pruned, output_dir=out)
    cfg.DATASETS.TRAIN=(RETAIN_DS,); cfg.DATASETS.TEST=(); cfg.DATALOADER.NUM_WORKERS=2
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE; cfg.SOLVER.BASE_LR=lr; cfg.SOLVER.MAX_ITER=iters; cfg.SOLVER.STEPS=[]
    Tr = make_ftsam_trainer(holder, ewc_lambda, rho); tr = Tr(cfg); tr.resume_or_load(resume=False); tr.train()
    return str(out/'model_final.pth')
print('Máy de-poison NB4 sẵn sàng (prune + FT-SAM + retain).')

Máy de-poison NB4 sẵn sàng (prune + FT-SAM + retain).


### 8b. Nạp s1: tìm weight có sẵn, nếu KHÔNG có thì tự train

In [11]:
import glob, os
# 1) Ưu tiên tìm weight s1 đã lưu (Kaggle Dataset / working). 2) Không thấy -> TỰ TRAIN s1.
S1_WEIGHTS = None
_cands = []
for pat in ['/kaggle/input/**/s1/model_final.pth', '/kaggle/input/**/*s1*.pth',
            '/kaggle/working/**/s1/model_final.pth', '/kaggle/working/s1/model_final.pth']:
    _cands += glob.glob(pat, recursive=True)
_cands = [c for c in sorted(set(_cands)) if 'poisoned_model' not in c]
print('Ứng viên weight s1:', _cands if _cands else '(không có)')
if _cands:
    S1_WEIGHTS = _cands[0]; print('-> Dùng weight có sẵn:', S1_WEIGHTS)
else:
    print('-> Không thấy weight s1. TỰ TRAIN lại cấu hình s1 của NB4 (prune5% + FT-SAM)...')
    S1_WEIGHTS = train_ftsam(prune_frac=0.05, n_retain=300, ewc_lambda=2000,
                             rho=0.05, lr=1e-4, iters=200, seed=42, tag='s1')
    print('-> Đã train xong s1:', S1_WEIGHTS)
assert S1_WEIGHTS is not None, 'Vẫn không có S1_WEIGHTS sau khi train.'

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Ứng viên weight s1: (không có)
-> Không thấy weight s1. TỰ TRAIN lại cấu hình s1 của NB4 (prune5% + FT-SAM)...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 19:14:42 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-08 19:15:11.379199: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783538111.594614      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783538111.655421      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783538112.147428      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783538112.147465      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783538112.147467      23 computation_placer.cc:177] computation placer alr

[07/08 19:15:50 d2.utils.events]:  eta: 0:03:21  iter: 39      time: 1.2658  last_time: 1.3647   lr: 1.958e-05  max_mem: 1305M
[07/08 19:16:18 d2.utils.events]:  eta: 0:03:04  iter: 59      time: 1.3220  last_time: 1.3905   lr: 2.9571e-05  max_mem: 1306M
[07/08 19:16:45 d2.utils.events]:  eta: 0:02:38  iter: 79      time: 1.3238  last_time: 1.2960   lr: 3.9561e-05  max_mem: 1306M
[07/08 19:17:11 d2.utils.events]:  eta: 0:02:11  iter: 99      time: 1.3222  last_time: 1.3394   lr: 4.955e-05  max_mem: 1306M
[07/08 19:17:39 d2.utils.events]:  eta: 0:01:46  iter: 119      time: 1.3302  last_time: 1.3794   lr: 5.954e-05  max_mem: 1306M
[07/08 19:18:06 d2.utils.events]:  eta: 0:01:20  iter: 139      time: 1.3341  last_time: 1.3374   lr: 6.9531e-05  max_mem: 1306M
[07/08 19:18:33 d2.utils.events]:  eta: 0:00:53  iter: 159      time: 1.3343  last_time: 1.3361   lr: 7.9521e-05  max_mem: 1306M
[07/08 19:19:00 d2.utils.events]:  eta: 0:00:26  iter: 179      time: 1.3359  last_time: 1.3494   lr: 8.

## 9. Xác nhận 2 model: firing/retention ở CAND_THRESH (0.05)

Kiểm tra NB4-s1 đúng là bản de-poison thật (firing thấp hơn hẳn poisoned) trước khi A/B.


In [12]:
poisoned_stats = eval_model_on_val(poisoned_predictor, VAL)
s1_predictor = build_predictor(S1_WEIGHTS, CAND_THRESH)
s1_stats = eval_model_on_val(s1_predictor, VAL)
print('POISONED :', {k: round(v,4) for k,v in poisoned_stats.items()})
print('NB4-s1   :', {k: round(v,4) for k,v in s1_stats.items()})
assert s1_stats['poison_firing_rate'] < poisoned_stats['poison_firing_rate'] - 0.2, \
    'NB4-s1 firing không thấp hơn poisoned rõ rệt — weight có thể sai. Kiểm tra lại S1_WEIGHTS.'
print('\n-> Xác nhận: NB4-s1 giảm firing THẬT so với poisoned. Sẵn sàng A/B.')


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 19:19:35 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/s1/model_final.pth ...
POISONED : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
NB4-s1   : {'poison_firing_rate': 0.2883, 'streak_retention': 0.5912, 'val_macadd': 1.0758}

-> Xác nhận: NB4-s1 giảm firing THẬT so với poisoned. Sẵn sàng A/B.


## 10. A/B threshold sweep có kiểm soát

Cùng grid threshold áp lên (A) poisoned và (B) NB4-s1. So val_macadd ở TỪNG threshold.
- Nếu B < A đều khắp → de-poison weight thật đóng góp thật (đúng tiêu chí BTC).
- gap = A - B (dương = lợi ích của de-poison thật).


In [13]:
def eval_at_thresh(predictor, val, thr):
    pk = {}
    for i, item in enumerate(val):
        b, s = predict_image(predictor, item['img']); keep = s > thr
        pk[str(i)] = (b[keep], s[keep]) if len(b) else (b, s)
    return macadd(CLEAN_REF, pk)

def stats_at_thresh(predictor, val, thr):
    fired=ft=kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img']); keep = s>thr
        b2,s2 = (b[keep],s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return fired/max(ft,1), kept/max(st,1), macadd(CLEAN_REF, pk)

GRID = [0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
rows=[]
for thr in GRID:
    fa,ra,va = stats_at_thresh(poisoned_predictor, VAL, thr)
    fb,rb,vb = stats_at_thresh(s1_predictor, VAL, thr)
    rows.append(dict(thr=thr, A_poisoned=round(va,4), B_s1=round(vb,4), gap=round(va-vb,4),
                     A_fire=round(fa,3), B_fire=round(fb,3), A_ret=round(ra,3), B_ret=round(rb,3)))
ab_df = pd.DataFrame(rows)
print(ab_df.to_string(index=False))
print('\n-> gap>0 nghĩa là NB4-s1 (de-poison thật) TỐT HƠN poisoned ở cùng threshold.')
best_b = ab_df.loc[ab_df['B_s1'].idxmin()]
print(f"\nB_s1 tốt nhất @thr={best_b['thr']}: val_macadd={best_b['B_s1']} (gap={best_b['gap']})")


 thr  A_poisoned   B_s1     gap  A_fire  B_fire  A_ret  B_ret
0.20      1.4549 1.0758  0.3791   0.969   0.288  1.000  0.591
0.25      1.3904 1.0572  0.3331   0.963   0.135  1.000  0.453
0.30      1.3626 1.0815  0.2811   0.951   0.061  0.962  0.321
0.35      1.3249 1.1110  0.2139   0.890   0.018  0.925  0.233
0.40      1.2414 1.1458  0.0956   0.804   0.012  0.874  0.176
0.45      1.2041 1.1778  0.0263   0.761   0.006  0.836  0.132
0.50      1.1824 1.2358 -0.0534   0.699   0.000  0.780  0.063
0.55      1.1949 1.2737 -0.0787   0.650   0.000  0.717  0.025
0.60      1.1798 1.2737 -0.0939   0.613   0.000  0.667  0.025

-> gap>0 nghĩa là NB4-s1 (de-poison thật) TỐT HƠN poisoned ở cùng threshold.

B_s1 tốt nhất @thr=0.25: val_macadd=1.0572 (gap=0.3331)


## 11. Kết luận A/B + chọn cấu hình nộp (đúng tiêu chí co-author)


In [14]:
mean_gap = ab_df['gap'].mean()
pos_gap = (ab_df['gap'] > 0).mean()
print(f'gap trung bình (A-B) = {mean_gap:.4f} | tỉ lệ threshold có gap>0 = {pos_gap:.0%}')
if mean_gap > 0.02 and pos_gap >= 0.6:
    verdict = 'DE-POISON WEIGHT THẬT đóng góp rõ — NB4-s1 vượt poisoned ở đa số threshold. Nộp bản B (đúng tiêu chí BTC).'
elif mean_gap > 0:
    verdict = 'De-poison thật đóng góp NHẸ. Nộp bản B (an toàn về review), ưu tiên threshold có gap>0.'
else:
    verdict = 'CẢNH BÁO: A≈B hoặc A tốt hơn — điểm chủ yếu từ threshold, KHÔNG từ de-poison. Cân nhắc rủi ro review.'
print('\nKẾT LUẬN:', verdict)

# Chọn threshold nộp: an toàn (0.35-0.5) + val_macadd thấp nhất trong dải đó cho model B
SAFE = ab_df[(ab_df['thr']>=0.35) & (ab_df['thr']<=0.5)]
pick = SAFE.loc[SAFE['B_s1'].idxmin()]
SUBMIT_MODEL = 'B_s1'; SUBMIT_WEIGHTS = S1_WEIGHTS; SUBMIT_THRESH_FINAL = float(pick['thr'])
print(f"\n-> NỘP: model=NB4-s1 (de-poison thật) @thr={SUBMIT_THRESH_FINAL} "
      f"| local val_macadd={pick['B_s1']} firing={pick['B_fire']} retention={pick['B_ret']}")
print('   (Chỉ chọn trong dải an toàn 0.35-0.5 để tránh phân kỳ local↔LB.)')


gap trung bình (A-B) = 0.1226 | tỉ lệ threshold có gap>0 = 67%

KẾT LUẬN: DE-POISON WEIGHT THẬT đóng góp rõ — NB4-s1 vượt poisoned ở đa số threshold. Nộp bản B (đúng tiêu chí BTC).

-> NỘP: model=NB4-s1 (de-poison thật) @thr=0.35 | local val_macadd=1.111 firing=0.018 retention=0.233
   (Chỉ chọn trong dải an toàn 0.35-0.5 để tránh phân kỳ local↔LB.)


## 12. Submission = output TRỰC TIẾP model NB4-s1 @ threshold đã chọn

Không boost, không rescoring — thuần khiết, minh bạch cho review.


In [15]:
E = (np.zeros((0,4),np.float32), np.zeros(0,np.float32))
sub_predictor = build_predictor(SUBMIT_WEIGHTS, CAND_THRESH)
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H))
        x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H)); w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred={}
for p in tqdm(test_files, desc='infer test'):
    img=load_image(p); b,s=predict_image(sub_predictor, img)
    keep=s>SUBMIT_THRESH_FINAL; pred[p.stem]=(b[keep],s[keep]) if len(b) else (b,s)
sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),E)))
sub['prediction_string']=sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string']=' '
sub.to_csv(SUBMISSION_PATH, index=False)
total=sum(len(s) for _,s in pred.values()); npred=int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | dets={total} ({total/len(test_files):.3f}/img) @thr={SUBMIT_THRESH_FINAL} | imgs w/ preds: {npred}')


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 19:21:56 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/s1/model_final.pth ...


infer test: 100%|██████████| 2000/2000 [04:19<00:00,  7.70it/s]

saved /kaggle/working/submission.csv | dets=163 (0.082/img) @thr=0.35 | imgs w/ preds: 155


## 13. Tổng kết


In [16]:
print('='*60)
print('A/B RESULT:')
print(ab_df.to_string(index=False))
print('='*60)
print('POISONED :', {k: round(v,4) for k,v in poisoned_stats.items()})
print('NB4-s1   :', {k: round(v,4) for k,v in s1_stats.items()})
print(f'gap trung bình (A-B) = {mean_gap:.4f}')
print('KẾT LUẬN:', verdict)
print(f'NỘP: NB4-s1 @thr={SUBMIT_THRESH_FINAL}')
print('='*60)
print('So mốc LB: poisoned~379 | empty~284 | NB6=254 | NB4-s1(thr0.2)=258.7 | mục tiêu: < 254')


A/B RESULT:
 thr  A_poisoned   B_s1     gap  A_fire  B_fire  A_ret  B_ret
0.20      1.4549 1.0758  0.3791   0.969   0.288  1.000  0.591
0.25      1.3904 1.0572  0.3331   0.963   0.135  1.000  0.453
0.30      1.3626 1.0815  0.2811   0.951   0.061  0.962  0.321
0.35      1.3249 1.1110  0.2139   0.890   0.018  0.925  0.233
0.40      1.2414 1.1458  0.0956   0.804   0.012  0.874  0.176
0.45      1.2041 1.1778  0.0263   0.761   0.006  0.836  0.132
0.50      1.1824 1.2358 -0.0534   0.699   0.000  0.780  0.063
0.55      1.1949 1.2737 -0.0787   0.650   0.000  0.717  0.025
0.60      1.1798 1.2737 -0.0939   0.613   0.000  0.667  0.025
POISONED : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
NB4-s1   : {'poison_firing_rate': 0.2883, 'streak_retention': 0.5912, 'val_macadd': 1.0758}
gap trung bình (A-B) = 0.1226
KẾT LUẬN: DE-POISON WEIGHT THẬT đóng góp rõ — NB4-s1 vượt poisoned ở đa số threshold. Nộp bản B (đúng tiêu chí BTC).
NỘP: NB4-s1 @thr=0.35
So mốc LB: poisone